# PRO3342 Modelagem e Simulação de Sistemas de Produção
## Projeto BVE - Modelo computacional: Barcaças

### Grupo G112

1. Francisco Rocha Freitas Brandão (18404341)
2. Inigo Anduaga (18404466)
3. Kepa Danis (18404719)
4. Vasco Oliveira (18440540)

Neste lab, o modelo conceitual desenvolvido no Lab 07, revisado com base nos dados do cenário-base das barcaças, constitui a referência para a implementação do modelo computacional das barcaças.

## 1. Objetivo

O modelo implementa o **submodelo 2 (Barcaças)** do modelo conceitual do grupo (`G112_Modelo_Conceitual`, versão 1.2, seções 2.2, 4, 5 e 6). Ele representa a frota da BVE em ciclo contínuo: navegação da base até um terminal, bombeio para o navio, retorno, eventual fila na base e carga até completar o tanque. Ainda não há navios nem PCO: o terminal de cada viagem é sorteado com probabilidade proporcional à taxa de chegada de navios compradores (*λ_j* · `prob_compra_j`), e o volume, pela distribuição de demanda do terminal. O objetivo é verificar a frota e a base isoladamente e medir a capacidade de entrega antes da integração com os navios (Lab 10).

| | Arquivo | Conteúdo |
|:--|:--|:--|
| Entrada | `entrada/frota.csv` | Uma linha por barcaça: capacidade, velocidade, taxa de bombeio, carga inicial |
| Entrada | `entrada/terminais.csv` | Parâmetros dos terminais (formato do submodelo 1) e distância à base `dist_base` |
| Entrada | `entrada/cenario.csv` | Configuração: data de início, horizonte, replicações, semente, fator de demanda, berços e taxa de carga da base, parâmetros do PCO (não usados aqui) |
| Saída | `saida/log_barcacas.csv` | Uma linha por estado concluído de cada barcaça |

Os três arquivos de entrada são gerados ou conferidos a partir dos dados do cenário-base (`entrada/D08_2_...md` para os navios e `entrada/D09_2_...md` para as barcaças, que revisa `prob_compra`). Como `terminais.csv` mantém o formato do submodelo 1, o mesmo arquivo servirá ao modelo integrado.

**Alterações do modelo conceitual (versão 1.2)**, registradas no controle de versões de `G112_Modelo_Conceitual`:

| Item | Versão 1.1 | Versão 1.2 (dados D09) |
|:--|:--|:--|
| Frota | 6 barcaças (carta-convite) | 4 barcaças de 3.000 t, 4,5 km/h, 200 t/h |
| Tempos | Conexão e desconexão aleatórias (1,5 h); manobra na base | Navegação, bombeio e carga determinísticos; conexão, desconexão e manobras desprezadas |
| Falhas | TEF e TR em `frota.csv` | Sem falhas; campos removidos |
| Estado inicial | Ciclo iniciava pela carga | Barcaças cheias na base, partem em t = 0 |
| Base | `base.csv` | Berços e taxa de carga em `cenario.csv` (configuração) |
| Distâncias | Só em `distancias.csv` | `dist_base` em `terminais.csv`; matriz completa reservada ao submodelo 3 |
| `prob_compra` | Dados D08 | Revisada pela BVE (D09) |
| Log das barcaças | Campos gerais | Campos `viagem` e `volume_t` acrescentados; estados `navegando`, `bombeando`, `fila_base`, `carregando` |

**Premissas e limitações:** cada viagem atende um navio e o navio está sempre disponível (a janela de operação só entra no Lab 10); estoque da base ilimitado; operação 24 h sem falhas; no fim do horizonte, os estados em curso não entram no log. Como as barcaças nunca ficam ociosas, o modelo mede a **capacidade máxima** da frota, não o atendimento da demanda.

## 2. Prompt

**Ferramenta de IA:** Claude (Anthropic), em interface de chat. A IA apoiou a revisão do modelo conceitual, a preparação das entradas, a escrita do modelo SimPy e dos testes e a montagem deste notebook. Os resultados apresentados vêm da execução do notebook.

```markdown
Atue como especialista em simulação de eventos discretos (SimPy). Implemente a atividade D09 - Projeto BVE:
Modelo Computacional das Barcaças (PRO3342), seguindo o modelo conceitual do grupo G112.

FONTES (ordem de prioridade)
1. Dados do cenário-base das barcaças (D09_2, Markdown) e dos navios (D08_2).
2. Modelo conceitual do grupo (versão 1.1): submodelo 2. Antes de implementar, liste o que os dados D09 alteram
   no modelo conceitual e proponha a versão 1.2 (registrar no controle de versões).
3. Notebook-modelo D09_1 (estrutura do relatório) e o modelo dos navios já implementado (D08), cujo estilo,
   funções de leitura e formato de testes devem ser mantidos.
Não inventar dados. Suposições só se imprescindíveis, sempre documentadas.

ENTRADAS (CSV; nenhum parâmetro do cenário fixo no código)
- frota.csv: id_barcaca, capacidade (t), velocidade (km/h), vazao_bomba (t/h), carga_inicial (t). Gerar a partir de D09.
- terminais.csv: formato do submodelo 1 (id_terminal, id_classe, n_bercos, taxa_chegada, antecedencia, tempo_manobra,
  dist_operacao, op_p1..p3, prob_compra, dist_volume, vl_p1..p3) + dist_base (km). Gerar de D08 com as revisões de D09.
- cenario.csv (configuração): id_cenario, data_inicio, horizonte (dias), n_rep, semente, fator_demanda,
  n_bercos_base, taxa_carga_base (t/h), horarios_programacao, horizonte_programacao, regra_pco.

PROCESSO (uma instância por barcaça k; ciclo contínuo)
- t = 0: todas as barcaças na base, tanque cheio, partem para o primeiro atendimento.
- Cada viagem: sortear terminal j com probabilidade proporcional a taxa_chegada_j * min(1, prob_compra_j * fator_demanda)
  e volume a ~ dist_volume_j; navegar dist_base_j / velocidade_k; bombear min(a, carga) / vazao_bomba_k;
  voltar; requisitar berço da base (n_bercos_base, FIFO); carregar (capacidade - carga) / taxa_carga_base.
- Tempos determinísticos; conexão, desconexão e manobras desprezadas; navio sempre disponível; sem falhas.
- Fluxos aleatórios independentes por barcaça e por variável (destino, volume), com SeedSequence([semente, rep]).

SAÍDA: log_barcacas.csv, uma linha por estado concluído: id_cenario, rep, id_barcaca, viagem, estado
(navegando, bombeando, fila_base, carregando), local (Base, Tj, Base-Tj, Tj-Base), t_inicio, t_fim (data-hora a partir
de data_inicio), carga_t (a bordo ao fim do estado), volume_t (bombeado ou carregado no estado).

EXECUÇÃO: uma replicação do cenário-base, 365 dias.

VERIFICAÇÃO (semente fixa; para cada teste: objetivo, configuração, esperado, obtido, conclusão; executar de fato)
Entradas; ciclo determinístico com uma barcaça, um terminal e demanda constante; estado inicial; tempos de navegação,
bombeio e carga; frequência dos destinos; volumes; capacidade e FIFO dos berços da base; frota heterogênea;
continuidade e ordem dos estados no log; conservação do combustível; formato do log; reprodutibilidade;
parametrização; comparação do ciclo médio e da utilização dos berços com o cálculo analítico.
```

**Iterações subsequentes:** (1) comparação dos dados D09 com o modelo conceitual 1.1 e redação da versão 1.2; (2) geração de `frota.csv` e `terminais.csv` a partir dos arquivos de dados; (3) implementação e primeira execução; o registro do estado `fila_base` passou a ser feito também com espera nula, para permitir o teste de FIFO; (4) escrita e execução dos testes; (5) cálculo da capacidade da frota frente à demanda esperada; (6) execução do notebook em kernel limpo.

## 3. Modelo de simulação

O fluxo é *dados → arquivos de entrada → leitura dos parâmetros → simulação → log*. Trocar de cenário (outra frota, outra base, outro horizonte) exige apenas alterar os arquivos de `entrada/`.

### 3.1 Arquivos de entrada

`terminais.csv` e `frota.csv` são gerados por funções que interpretam os arquivos de dados: os parâmetros dos navios vêm de D08 e são atualizados por D09 (`prob_compra` revisada e `dist_base`); a frota vem da tabela de D09, com uma linha por barcaça e `carga_inicial` igual à capacidade (tanque cheio). `cenario.csv` é editado diretamente, pois reúne escolhas do cenário; seus valores da base (berços e taxa de carga) são conferidos contra D09 no teste V01.

In [1]:
import re
from pathlib import Path

import pandas as pd

HORAS_POR_DIA = 24.0
ARQ_NAVIOS = "entrada/D08_2_BVE_MComp_Navios_-_Dados.md"
ARQ_BARCAS = "entrada/D09_2_BVE_MComp_Barcas_-_Dados.md"


def _num(txt):
    txt = txt.strip()
    if re.search(r"\d\.\d{3}\b", txt):
        txt = txt.replace(".", "")
    return float(txt.replace(",", "."))


def _secao(md, n):
    return re.search(rf"^## {n}\.[^\n]*\n(.*?)(?=^## \d+\.|\Z)", md, flags=re.S | re.M).group(1)


def _por_terminal(texto, padrao):
    return {m.group(1): m.groups()[1:] for m in re.finditer(rf"^- (T\d+): {padrao}\s*$", texto, flags=re.M)}


def _tabela(texto):
    linhas = [l for l in texto.splitlines() if l.startswith("|")]
    cab = [c.strip() for c in linhas[0].strip("|").split("|")]
    return cab, [[c.strip() for c in l.strip("|").split("|")] for l in linhas[2:]]


def _horas(txt):
    h, mi = re.search(r"(\d+)\s*h", txt), re.search(r"(\d+)\s*min", txt)
    return (int(h.group(1)) if h else 0) + (int(mi.group(1)) if mi else 0) / 60.0


def ler_dados_navios(caminho):
    md = Path(caminho).read_text(encoding="utf-8")
    tri = r"Tri\(([\d,]+),\s*([\d,]+),\s*([\d,]+)\)"
    return {
        "bercos": {k: int(v[0]) for k, v in _por_terminal(_secao(md, 1), r"(\d+) ber[çc]os?").items()},
        "taxa_dia": {k: _num(v[0]) for k, v in _por_terminal(_secao(md, 3), r"([\d,]+) navios?/dia").items()},
        "antecedencia": float(re.search(r"(\d+)\s*h", _secao(md, 5)).group(1)),
        "manobra": {k: _horas(v[0]) for k, v in _por_terminal(_secao(md, 7), r"(.+?)").items()},
        "operacao": {k: tuple(map(_num, v)) for k, v in _por_terminal(_secao(md, 8), tri + r"\s*h").items()},
        "prob": {k: _num(v[0]) / 100 for k, v in _por_terminal(_secao(md, 9), r"(\d+)%").items()},
        "volume": {k: tuple(map(_num, v)) for k, v in _por_terminal(_secao(md, 10), tri + r"\s*t").items()},
    }


def ler_dados_barcas(caminho):
    md = Path(caminho).read_text(encoding="utf-8")
    s3 = _secao(md, 3)
    _, carac = _tabela(s3)
    carac = {nome: _num(re.match(r"[\d.,]+", valor).group(0)) for nome, valor in carac}
    _, probs = _tabela(_secao(md, 6))
    cab, dist = _tabela(_secao(md, 8))
    return {
        "n_bercos_base": int(re.search(r"(\d+) ber[çc]os", _secao(md, 1)).group(1)),
        "taxa_carga_base": _num(re.search(r"([\d.,]+) t/h", _secao(md, 2)).group(1)),
        "n_barcacas": int(re.search(r"tem (\d+) barca", s3).group(1)),
        "capacidade": carac["Capacidade do tanque"],
        "velocidade": carac["Velocidade de navegação no canal"],
        "vazao_bomba": carac["Taxa de bombeio para o navio"],
        "tanque_cheio": "tanque cheio" in _secao(md, 4),
        "prob": {t: _num(p.rstrip("%")) / 100 for t, p in probs},
        "dist_base": {t: _num(v.replace("km", "")) for t, v in zip(cab[1:], dist[0][1:])},
    }


def montar_terminais(dn, db):
    linhas = []
    for t in sorted(dn["bercos"], key=lambda x: int(x[1:])):
        op, vl = dn["operacao"][t], dn["volume"][t]
        linhas.append({
            "id_terminal": t, "id_classe": "C" + t[1:], "n_bercos": dn["bercos"][t],
            "taxa_chegada": dn["taxa_dia"][t] / HORAS_POR_DIA, "antecedencia": dn["antecedencia"],
            "tempo_manobra": dn["manobra"][t],
            "dist_operacao": "triangular", "op_p1": op[0], "op_p2": op[1], "op_p3": op[2],
            "prob_compra": db["prob"].get(t, dn["prob"][t]),
            "dist_volume": "triangular", "vl_p1": vl[0], "vl_p2": vl[1], "vl_p3": vl[2],
            "dist_base": db["dist_base"][t],
        })
    return pd.DataFrame(linhas)


def montar_frota(db):
    return pd.DataFrame([{
        "id_barcaca": f"B{k}", "capacidade": db["capacidade"], "velocidade": db["velocidade"],
        "vazao_bomba": db["vazao_bomba"], "carga_inicial": db["capacidade"] if db["tanque_cheio"] else 0.0,
    } for k in range(1, db["n_barcacas"] + 1)])


def preparar_entradas(pasta):
    pasta = Path(pasta)
    dn, db = ler_dados_navios(ARQ_NAVIOS), ler_dados_barcas(ARQ_BARCAS)
    montar_terminais(dn, db).to_csv(pasta / "terminais.csv", index=False)
    montar_frota(db).to_csv(pasta / "frota.csv", index=False)

In [2]:
preparar_entradas("entrada")
display(pd.read_csv("entrada/frota.csv"))
display(pd.read_csv("entrada/terminais.csv"))
display(pd.read_csv("entrada/cenario.csv"))

,id_barcaca,capacidade,velocidade,vazao_bomba,carga_inicial
0,B1,3000.0,4.5,200.0,3000.0
1,B2,3000.0,4.5,200.0,3000.0
2,B3,3000.0,4.5,200.0,3000.0
3,B4,3000.0,4.5,200.0,3000.0


,id_terminal,id_classe,n_bercos,taxa_chegada,antecedencia,tempo_manobra,dist_operacao,op_p1,op_p2,op_p3,prob_compra,dist_volume,vl_p1,vl_p2,vl_p3,dist_base
0,T1,C1,3,0.104167,48.0,1.666667,triangular,12.0,15.0,19.0,0.63,triangular,600.0,1000.0,1800.0,2.1
1,T2,C2,5,0.166667,48.0,1.500000,triangular,14.0,18.0,22.0,0.54,triangular,800.0,1200.0,2000.0,3.7
2,T3,C3,4,0.187500,48.0,1.333333,triangular,9.0,12.0,15.0,0.45,triangular,400.0,700.0,1200.0,5.8
3,T4,C4,3,0.145833,48.0,1.333333,triangular,12.0,15.0,18.0,0.59,triangular,500.0,900.0,1600.0,7.5
4,T5,C5,2,0.083333,48.0,1.083333,triangular,9.0,12.0,15.0,0.50,triangular,300.0,600.0,1000.0,9.2
5,T6,C6,1,0.041667,48.0,0.916667,triangular,8.0,10.0,12.0,0.45,triangular,250.0,500.0,900.0,10.6


,id_cenario,data_inicio,horizonte,n_rep,semente,fator_demanda,n_bercos_base,taxa_carga_base,horarios_programacao,horizonte_programacao,regra_pco
0,BASE,2026-01-01 00:00:00,365,1,20260,1.0,2,400,09:30;16:30,36,menor_fim_janela


### 3.2 Modelo SimPy

Principais decisões de implementação:

- **Classe `Frota`.** Como sugerido na aula 9, a simulação é encapsulada em uma classe que reúne o recurso (berços da base, um `simpy.Resource` com capacidade `n_bercos_base`), as entidades (barcaças, com seus atributos lidos de `frota.csv`) e os processos (sorteio do atendimento, navegação, bombeio, fila e carga). A fila do `Resource` é FIFO.
- **Sorteio do atendimento.** Pesos *λ_j* · min(1, `prob_compra_j` · *f_d*), normalizados; volume pela distribuição do terminal. Cada barcaça tem dois fluxos aleatórios próprios (destino e volume), derivados de `SeedSequence([semente, rep])`; assim, alterar uma barcaça não muda os sorteios das demais.
- **Tempos.** Calculados a partir dos atributos: navegação = `dist_base` / `velocidade`; bombeio = volume / `vazao_bomba`; carga = (`capacidade` − carga a bordo) / `taxa_carga_base`.
- **Log.** Ao fim de cada estado grava-se uma linha. O estado `fila_base` é gravado mesmo com espera nula, o que preserva a continuidade do log (o fim de um estado é o início do seguinte) e permite verificar a disciplina FIFO. Estados em curso no fim do horizonte não são gravados.
- **Volume maior que a carga.** A barcaça entrega o que tem a bordo. No cenário-base isso não ocorre, pois o maior volume (2.000 t) é menor que a capacidade (3.000 t) e a barcaça sempre parte cheia.

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import simpy

HORAS_POR_DIA = 24.0
COLUNAS_LOG = ["id_cenario", "rep", "id_barcaca", "viagem", "estado", "local",
               "t_inicio", "t_fim", "carga_t", "volume_t"]


def carregar_parametros(pasta, id_cenario=None):
    pasta = Path(pasta)
    frota = pd.read_csv(pasta / "frota.csv")
    terminais = pd.read_csv(pasta / "terminais.csv")
    cenarios = pd.read_csv(pasta / "cenario.csv", parse_dates=["data_inicio"])
    cenario = cenarios.iloc[0] if id_cenario is None else cenarios.loc[cenarios["id_cenario"] == id_cenario].iloc[0]
    return frota, terminais, cenario


def sortear(rng, dist, p1, p2=None, p3=None):
    if dist == "constante":
        return float(p1)
    if dist == "uniforme":
        return rng.uniform(p1, p2)
    if dist == "exponencial":
        return rng.exponential(p1)
    if dist == "triangular":
        return rng.triangular(p1, p2, p3)
    if dist == "erlang":
        return rng.gamma(p1, p2)
    if dist == "lognormal":
        return rng.lognormal(p1, p2)
    raise ValueError(f"Distribuição desconhecida: {dist}")


class Barcaca:
    def __init__(self, linha, sequencias):
        self.id = linha["id_barcaca"]
        self.capacidade = float(linha["capacidade"])
        self.velocidade = float(linha["velocidade"])
        self.vazao_bomba = float(linha["vazao_bomba"])
        self.carga = float(linha["carga_inicial"])
        self.rng_destino, self.rng_volume = (np.random.default_rng(s) for s in sequencias)


class Frota:
    def __init__(self, frota, terminais, cenario, rep):
        self.cenario, self.rep = cenario, int(rep)
        self.env = simpy.Environment()
        self.horizonte_h = float(cenario["horizonte"]) * HORAS_POR_DIA
        self.bercos = simpy.Resource(self.env, capacity=int(cenario["n_bercos_base"]))
        self.taxa_carga = float(cenario["taxa_carga_base"])
        self.terminais = terminais.reset_index(drop=True)
        prob = np.minimum(1.0, self.terminais["prob_compra"].to_numpy() * float(cenario["fator_demanda"]))
        pesos = self.terminais["taxa_chegada"].to_numpy() * prob
        self.p_destino = pesos / pesos.sum()
        n_fluxos = 2  # destino, volume
        sementes = np.random.SeedSequence([int(cenario["semente"]), self.rep]).spawn(n_fluxos * len(frota))
        self.barcacas = [Barcaca(l, sementes[n_fluxos * i:n_fluxos * (i + 1)])
                         for i, (_, l) in enumerate(frota.iterrows())]
        self.registros = []
        self.ocupacao_max = 0
        self.fila_max = 0

    def registrar(self, b, viagem, estado, local, t_inicio, volume):
        self.registros.append({"id_cenario": self.cenario["id_cenario"], "rep": self.rep, "id_barcaca": b.id,
                               "viagem": viagem, "estado": estado, "local": local, "t_inicio": t_inicio,
                               "t_fim": self.env.now, "carga_t": b.carga, "volume_t": volume})

    def sortear_atendimento(self, b):
        j = b.rng_destino.choice(len(self.terminais), p=self.p_destino)
        t = self.terminais.iloc[j]
        volume = sortear(b.rng_volume, t["dist_volume"], t["vl_p1"], t["vl_p2"], t["vl_p3"])
        return t["id_terminal"], float(t["dist_base"]), volume

    def ciclo(self, b):
        env, viagem = self.env, 0
        while True:
            viagem += 1
            terminal, dist, volume = self.sortear_atendimento(b)
            t0 = env.now
            yield env.timeout(dist / b.velocidade)
            self.registrar(b, viagem, "navegando", f"Base-{terminal}", t0, 0.0)
            entregue, t0 = min(volume, b.carga), env.now
            yield env.timeout(entregue / b.vazao_bomba)
            b.carga -= entregue
            self.registrar(b, viagem, "bombeando", terminal, t0, entregue)
            t0 = env.now
            yield env.timeout(dist / b.velocidade)
            self.registrar(b, viagem, "navegando", f"{terminal}-Base", t0, 0.0)
            t0 = env.now
            with self.bercos.request() as req:
                self.fila_max = max(self.fila_max, len(self.bercos.queue))
                yield req
                self.registrar(b, viagem, "fila_base", "Base", t0, 0.0)
                self.ocupacao_max = max(self.ocupacao_max, self.bercos.count)
                carga_falta, t0 = b.capacidade - b.carga, env.now
                yield env.timeout(carga_falta / self.taxa_carga)
                b.carga = b.capacidade
                self.registrar(b, viagem, "carregando", "Base", t0, carga_falta)

    def simular(self):
        for b in self.barcacas:
            self.env.process(self.ciclo(b))
        self.env.run(until=self.horizonte_h)
        diag = {"ocupacao_max": self.ocupacao_max, "fila_max": self.fila_max,
                "carga_final": {b.id: b.carga for b in self.barcacas}, "p_destino": self.p_destino}
        return montar_log(self.registros, self.cenario), diag


def montar_log(registros, cenario):
    log = pd.DataFrame(registros, columns=COLUNAS_LOG)
    origem = pd.Timestamp(cenario["data_inicio"])
    for c in ("t_inicio", "t_fim"):
        log[c] = (origem + pd.to_timedelta(log[c], unit="h")).dt.round("us")
    log = log.sort_values(["id_barcaca", "t_inicio", "t_fim"], kind="stable").reset_index(drop=True)
    log["rep"] = log["rep"].astype(int)
    log["viagem"] = log["viagem"].astype(int)
    return log


def simular(frota, terminais, cenario, rep):
    return Frota(frota, terminais, cenario, rep).simular()


def gravar_log(log, caminho):
    Path(caminho).parent.mkdir(parents=True, exist_ok=True)
    log.to_csv(caminho, index=False, date_format="%Y-%m-%d %H:%M:%S.%f")


def executar(pasta_entrada, pasta_saida, id_cenario=None):
    frota, terminais, cenario = carregar_parametros(pasta_entrada, id_cenario)
    logs, diags = [], []
    for rep in range(1, int(cenario["n_rep"]) + 1):
        log, diag = simular(frota, terminais, cenario, rep)
        logs.append(log)
        diags.append(diag)
    log = pd.concat(logs, ignore_index=True)
    gravar_log(log, Path(pasta_saida) / "log_barcacas.csv")
    return log, diags

## 4. Saída

Replicação do cenário-base (365 dias), lida dos arquivos de entrada e gravada em `saida/log_barcacas.csv`.

In [4]:
frota, terminais, cenario = carregar_parametros("entrada")
print("Cenário:", cenario["id_cenario"], "| horizonte (dias):", cenario["horizonte"], "| replicações:", cenario["n_rep"],
      "| semente:", cenario["semente"], "| barcaças:", len(frota), "| berços da base:", cenario["n_bercos_base"])

log_base, diagnosticos = executar("entrada", "saida")
diag_base = diagnosticos[0]
print("Linhas no log:", len(log_base))
display(log_base.head(12))

Cenário: BASE | horizonte (dias): 365 | replicações: 1 | semente: 20260 | barcaças: 4 | berços da base: 2


Linhas no log: 17383


,id_cenario,rep,id_barcaca,viagem,estado,local,t_inicio,t_fim,carga_t,volume_t
0,BASE,1,B1,1,navegando,Base-T2,2026-01-01 00:00:00.000000,2026-01-01 00:49:20.000000,3000.000000,0.000000
1,BASE,1,B1,1,bombeando,T2,2026-01-01 00:49:20.000000,2026-01-01 06:42:15.579721,1823.578904,1176.421096
2,BASE,1,B1,1,navegando,T2-Base,2026-01-01 06:42:15.579721,2026-01-01 07:31:35.579721,1823.578904,0.000000
3,BASE,1,B1,1,fila_base,Base,2026-01-01 07:31:35.579721,2026-01-01 07:31:35.579721,1823.578904,0.000000
4,BASE,1,B1,1,carregando,Base,2026-01-01 07:31:35.579721,2026-01-01 10:28:03.369582,3000.000000,1176.421096
5,BASE,1,B1,2,navegando,Base-T4,2026-01-01 10:28:03.369582,2026-01-01 12:08:03.369582,3000.000000,0.000000
6,BASE,1,B1,2,bombeando,T4,2026-01-01 12:08:03.369582,2026-01-01 15:41:11.239958,2289.562757,710.437243
7,BASE,1,B1,2,navegando,T4-Base,2026-01-01 15:41:11.239958,2026-01-01 17:21:11.239958,2289.562757,0.000000
8,BASE,1,B1,2,fila_base,Base,2026-01-01 17:21:11.239958,2026-01-01 17:21:11.239958,2289.562757,0.000000
9,BASE,1,B1,2,carregando,Base,2026-01-01 17:21:11.239958,2026-01-01 19:07:45.175147,3000.000000,710.437243


In [5]:
def _h(s):
    return s.dt.total_seconds() / 3600

H_base = float(cenario["horizonte"]) * HORAS_POR_DIA
lb = log_base.assign(duracao_h=_h(log_base["t_fim"] - log_base["t_inicio"]))
por_barcaca = lb.pivot_table(index="id_barcaca", columns="estado", values="duracao_h", aggfunc="sum") / H_base
por_barcaca = por_barcaca[["navegando", "bombeando", "fila_base", "carregando"]].add_prefix("frac_")
por_barcaca.insert(0, "volume_entregue_t", lb[lb["estado"] == "bombeando"].groupby("id_barcaca")["volume_t"].sum())
por_barcaca.insert(0, "atendimentos", lb[lb["estado"] == "bombeando"].groupby("id_barcaca").size())
display(por_barcaca.round(3))

bomb = lb[lb["estado"] == "bombeando"]
car = lb[lb["estado"] == "carregando"]
fila = lb[lb["estado"] == "fila_base"]
ciclo = lb.groupby(["id_barcaca", "viagem"]).filter(lambda g: len(g) == 5).groupby(["id_barcaca", "viagem"])["duracao_h"].sum()
tp = terminais.assign(p=np.minimum(1, terminais["prob_compra"] * float(cenario["fator_demanda"])),
                      ea=(terminais["vl_p1"] + terminais["vl_p2"] + terminais["vl_p3"]) / 3)
demanda_dia = (tp["taxa_chegada"] * HORAS_POR_DIA * tp["p"] * tp["ea"]).sum()
pedidos_dia = (tp["taxa_chegada"] * HORAS_POR_DIA * tp["p"]).sum()
indicadores = pd.Series({
    "Atendimentos da frota por dia": len(bomb) / float(cenario["horizonte"]),
    "Volume entregue pela frota (t/dia)": bomb["volume_t"].sum() / float(cenario["horizonte"]),
    "Tempo de ciclo médio, com fila (h)": ciclo.mean(),
    "Espera média por berço na base (h)": fila["duracao_h"].mean(),
    "Fração de chegadas à base com espera": (fila["duracao_h"] > 1e-9).mean(),
    "Utilização dos berços da base": car["duracao_h"].sum() / (int(cenario["n_bercos_base"]) * H_base),
    "Pedidos esperados por dia (navios compradores)": pedidos_dia,
    "Demanda esperada (t/dia)": demanda_dia,
    "Capacidade da frota / demanda esperada (volume)": bomb["volume_t"].sum() / float(cenario["horizonte"]) / demanda_dia,
})
display(indicadores.round(3).to_frame("valor"))

estado,atendimentos,volume_entregue_t,frac_navegando,frac_bombeando,frac_fila_base,frac_carregando
id_barcaca,,,,,,
B1,864,865470.544,0.246,0.494,0.012,0.247
B2,876,852981.324,0.254,0.487,0.015,0.243
B3,870,857895.947,0.248,0.490,0.018,0.244
B4,867,866626.810,0.247,0.495,0.011,0.247


,valor
Atendimentos da frota por dia,9.526
Volume entregue pela frota (t/dia),9432.807
"Tempo de ciclo médio, com fila (h)",10.076
Espera média por berço na base (h),0.142
Fração de chegadas à base com espera,0.154
Utilização dos berços da base,0.491
Pedidos esperados por dia (navios compradores),9.275
Demanda esperada (t/dia),9163.333
Capacidade da frota / demanda esperada (volume),1.029


O trecho do log mostra os primeiros estados de cada barcaça: no instante 0 as quatro partem cheias da base (`navegando`, `Base-Tj`), bombeiam no terminal sorteado, voltam e, após a fila (em geral de duração nula), carregam o volume entregue. A tabela por barcaça mostra a fração do tempo em cada estado; os indicadores da frota são calculados a partir do log, conforme a seção 6 do modelo conceitual.

Como as barcaças nunca ficam ociosas neste submodelo, o volume entregue é a **capacidade máxima** da frota no cenário-base. A última linha compara essa capacidade com a demanda esperada dos navios compradores (Σ *λ_j* · `prob_compra_j` · E[*a_j*]): a folga é pequena, o que indica que a frota de 4 barcaças estará próxima do limite quando for integrada à demanda real no Lab 10, antes mesmo do efeito das janelas de atendimento.

**Campos do `log_barcacas.csv`** (uma linha por estado concluído):

| Campo | Descrição |
|:--|:--|
| `id_cenario`, `rep` | Cenário e replicação |
| `id_barcaca` | Barcaça (de `frota.csv`) |
| `viagem` | Número da viagem da barcaça; uma viagem = ida, bombeio, retorno, fila e carga |
| `estado` | `navegando`, `bombeando`, `fila_base` ou `carregando` |
| `local` | `Base`, terminal (`Tj`) ou trecho (`Base-Tj`, `Tj-Base`) |
| `t_inicio`, `t_fim` | Início e fim do estado, em data-hora a partir de `data_inicio` |
| `carga_t` | Combustível a bordo ao fim do estado (t) |
| `volume_t` | Volume bombeado (`bombeando`) ou carregado (`carregando`) no estado (t); 0 nos demais |

## 5. Verificação do modelo

Todos os testes usam a semente de `cenario.csv`. Os cenários de teste alteram em memória as tabelas lidas dos arquivos (horizontes curtos, uma única barcaça, um único terminal de destino, demanda constante, base com um berço, frota heterogênea), sem mudar a lógica do modelo. Os critérios estatísticos usam tolerância de 3 desvios-padrão. Cada teste registra objetivo, configuração, resultado esperado, resultado obtido e conclusão; a tabela final resume os resultados.

In [6]:
from IPython.display import Markdown

frota_base, terminais_base, cenario_base = carregar_parametros("entrada")
origem = pd.Timestamp(cenario_base["data_inicio"])
RESULTADOS = []
TOL = 1e-8


def cen(**kw):
    c = cenario_base.copy()
    for k, v in kw.items():
        c[k] = v
    return c


def horas(serie):
    return serie.dt.total_seconds() / 3600


def relativo(serie):
    return horas(serie - origem)


def dur(log):
    return horas(log["t_fim"] - log["t_inicio"])


def relatar(codigo, nome, objetivo, config, esperado, obtido, ok, conclusao):
    ok = bool(ok)
    RESULTADOS.append({"Teste": codigo, "Descrição": nome, "Resultado": "Aprovado" if ok else "Reprovado"})
    display(Markdown(
        f"**{codigo} – {nome}**\n\n"
        f"- **Objetivo:** {objetivo}\n"
        f"- **Configuração:** {config}\n"
        f"- **Esperado:** {esperado}\n"
        f"- **Obtido:** {obtido}\n"
        f"- **Conclusão:** {'Aprovado. ' if ok else 'REPROVADO. '}{conclusao}"))


def so_terminal(terminais, j):
    t = terminais.copy()
    t.loc[t["id_terminal"] != j, "prob_compra"] = 0.0
    return t


def volume_constante(terminais, valor):
    t = terminais.copy()
    t["dist_volume"], t["vl_p1"] = "constante", valor
    return t


def ocupacao_maxima(log):
    c = log[log["estado"] == "carregando"]
    ev = sorted([(t, 1) for t in c["t_inicio"]] + [(t, -1) for t in c["t_fim"]], key=lambda x: (x[0], x[1]))
    atual = maximo = 0
    for _, d in ev:
        atual += d
        maximo = max(maximo, atual)
    return maximo

In [7]:
# V01 - Arquivos de entrada
dn, db = ler_dados_navios(ARQ_NAVIOS), ler_dados_barcas(ARQ_BARCAS)
try:
    pd.testing.assert_frame_equal(pd.read_csv("entrada/terminais.csv"), montar_terminais(dn, db), check_exact=False, atol=1e-12)
    pd.testing.assert_frame_equal(pd.read_csv("entrada/frota.csv"), montar_frota(db), check_exact=False, atol=1e-12)
    iguais = True
except AssertionError:
    iguais = False
prob_rev = dict(zip(terminais_base["id_terminal"], terminais_base["prob_compra"])) == db["prob"]
base_ok = int(cenario_base["n_bercos_base"]) == db["n_bercos_base"] and float(cenario_base["taxa_carga_base"]) == db["taxa_carga_base"]
campos_cen = ["id_cenario", "data_inicio", "horizonte", "n_rep", "semente", "fator_demanda", "n_bercos_base",
              "taxa_carga_base", "horarios_programacao", "horizonte_programacao", "regra_pco"]
cen_ok = list(pd.read_csv("entrada/cenario.csv").columns) == campos_cen
relatar("V01", "Arquivos de entrada",
        "Confirmar que frota, terminais e configuração reproduzem os dados do cenário-base e seguem o formato do modelo conceitual 1.2.",
        "`frota.csv` e `terminais.csv` relidos e comparados com nova leitura dos arquivos de dados; `prob_compra` comparada com a revisão de D09; base e campos de `cenario.csv`.",
        f"Arquivos idênticos à leitura dos dados; {db['n_barcacas']} barcaças; `prob_compra` igual à revisão de D09; `n_bercos_base` = {db['n_bercos_base']} e `taxa_carga_base` = {db['taxa_carga_base']:g} t/h; 11 campos em `cenario.csv`; horizonte de 365 dias.",
        f"arquivos idênticos: {iguais}; barcaças: {len(frota_base)}; `prob_compra` revisada: {prob_rev}; base conforme D09: {base_ok}; campos de `cenario.csv`: {cen_ok}; horizonte: {cenario_base['horizonte']} dias.",
        iguais and prob_rev and base_ok and cen_ok and len(frota_base) == db["n_barcacas"] and float(cenario_base["horizonte"]) == 365,
        "Os parâmetros do cenário-base vêm dos arquivos de dados e de `cenario.csv`, no formato do modelo conceitual.")

**V01 – Arquivos de entrada**

- **Objetivo:** Confirmar que frota, terminais e configuração reproduzem os dados do cenário-base e seguem o formato do modelo conceitual 1.2.
- **Configuração:** `frota.csv` e `terminais.csv` relidos e comparados com nova leitura dos arquivos de dados; `prob_compra` comparada com a revisão de D09; base e campos de `cenario.csv`.
- **Esperado:** Arquivos idênticos à leitura dos dados; 4 barcaças; `prob_compra` igual à revisão de D09; `n_bercos_base` = 2 e `taxa_carga_base` = 400 t/h; 11 campos em `cenario.csv`; horizonte de 365 dias.
- **Obtido:** arquivos idênticos: True; barcaças: 4; `prob_compra` revisada: True; base conforme D09: True; campos de `cenario.csv`: True; horizonte: 365 dias.
- **Conclusão:** Aprovado. Os parâmetros do cenário-base vêm dos arquivos de dados e de `cenario.csv`, no formato do modelo conceitual.

In [8]:
# V02 - Ciclo determinístico: uma barcaça, um terminal, demanda constante
j, a, H10 = "T1", 1000.0, 10
f1 = frota_base.iloc[[0]].copy()
log2, _ = simular(f1, volume_constante(so_terminal(terminais_base, j), a), cen(horizonte=H10), 1)
r = f1.iloc[0]
u = float(terminais_base.set_index("id_terminal").loc[j, "dist_base"])
esp = {"navegando": u / r["velocidade"], "bombeando": a / r["vazao_bomba"], "fila_base": 0.0, "carregando": a / float(cenario_base["taxa_carga_base"])}
ciclo_esp = 2 * esp["navegando"] + esp["bombeando"] + esp["carregando"]
n_esp = int(H10 * HORAS_POR_DIA // ciclo_esp)
err = max((dur(log2[log2["estado"] == e]) - v).abs().max() for e, v in esp.items())
n_obt = int((log2["estado"] == "carregando").sum())
destinos_ok = set(log2.loc[log2["estado"] == "bombeando", "local"]) == {j}
relatar("V02", "Ciclo determinístico",
        "Confirmar o encadeamento e a duração dos estados num caso em que o resultado pode ser calculado à mão.",
        f"Uma barcaça ({r['id_barcaca']}), destino sempre {j} ({u:g} km), volume constante de {a:g} t, {H10} dias.",
        f"Navegação {esp['navegando']:.4f} h por trecho, bombeio {esp['bombeando']:.2f} h, carga {esp['carregando']:.2f} h, fila nula; ciclo de {ciclo_esp:.4f} h; {n_esp} cargas concluídas em {H10*24} h; todos os bombeios em {j}.",
        f"Erro máximo nas durações: {err:.1e} h; cargas concluídas: {n_obt}; destinos: {sorted(set(log2.loc[log2['estado'] == 'bombeando', 'local']))}.",
        err <= TOL and n_obt == n_esp and destinos_ok,
        "O ciclo navegação–bombeio–retorno–carga e os tempos calculados a partir dos atributos estão corretos.")

**V02 – Ciclo determinístico**

- **Objetivo:** Confirmar o encadeamento e a duração dos estados num caso em que o resultado pode ser calculado à mão.
- **Configuração:** Uma barcaça (B1), destino sempre T1 (2.1 km), volume constante de 1000 t, 10 dias.
- **Esperado:** Navegação 0.4667 h por trecho, bombeio 5.00 h, carga 2.50 h, fila nula; ciclo de 8.4333 h; 28 cargas concluídas em 240 h; todos os bombeios em T1.
- **Obtido:** Erro máximo nas durações: 0.0e+00 h; cargas concluídas: 28; destinos: ['T1'].
- **Conclusão:** Aprovado. O ciclo navegação–bombeio–retorno–carga e os tempos calculados a partir dos atributos estão corretos.

In [9]:
# V03 - Estado inicial
prim = log_base.sort_values("t_inicio").groupby("id_barcaca").head(2).sort_values(["id_barcaca", "t_inicio"])
p1 = prim.groupby("id_barcaca").nth(0)
p2 = prim.groupby("id_barcaca").nth(1)
ok_ini = ((p1["estado"] == "navegando").all() and (relativo(p1["t_inicio"]).abs() < TOL).all()
          and p1["local"].str.startswith("Base-").all() and (p1["viagem"] == 1).all()
          and np.allclose(p1["carga_t"].values, frota_base.set_index("id_barcaca").loc[p1["id_barcaca"], "carga_inicial"].values)
          and (p2["estado"] == "bombeando").all())
relatar("V03", "Estado inicial",
        "Verificar que as barcaças começam na base, com o tanque cheio, e partem em t = 0 sem carregar antes.",
        "Primeiros dois estados de cada barcaça no cenário-base.",
        "Para todas as barcaças: primeiro estado `navegando` saindo da base em t = 0, viagem 1, carga igual a `carga_inicial`; segundo estado `bombeando`.",
        f"{len(p1)} barcaças; primeiros estados: {sorted(set(p1['estado']))}, início máximo {relativo(p1['t_inicio']).max():.1e} h, cargas {sorted(set(p1['carga_t']))}; segundos estados: {sorted(set(p2['estado']))}.",
        ok_ini and len(p1) == len(frota_base),
        "A condição inicial dos dados D09 está implementada.")

**V03 – Estado inicial**

- **Objetivo:** Verificar que as barcaças começam na base, com o tanque cheio, e partem em t = 0 sem carregar antes.
- **Configuração:** Primeiros dois estados de cada barcaça no cenário-base.
- **Esperado:** Para todas as barcaças: primeiro estado `navegando` saindo da base em t = 0, viagem 1, carga igual a `carga_inicial`; segundo estado `bombeando`.
- **Obtido:** 4 barcaças; primeiros estados: ['navegando'], início máximo 0.0e+00 h, cargas [3000.0]; segundos estados: ['bombeando'].
- **Conclusão:** Aprovado. A condição inicial dos dados D09 está implementada.

In [10]:
# V04 - Tempos de navegação, bombeio e carga e carga a bordo
fb = frota_base.set_index("id_barcaca")
dist = terminais_base.set_index("id_terminal")["dist_base"]
nav = log_base[log_base["estado"] == "navegando"]
term_nav = nav["local"].str.replace("Base", "").str.strip("-")
e_nav = (dur(nav) - term_nav.map(dist).values / fb.loc[nav["id_barcaca"], "velocidade"].values).abs().max()
bo = log_base[log_base["estado"] == "bombeando"]
e_bom = (dur(bo) - bo["volume_t"].values / fb.loc[bo["id_barcaca"], "vazao_bomba"].values).abs().max()
ca = log_base[log_base["estado"] == "carregando"]
e_car = (dur(ca) - ca["volume_t"] / float(cenario_base["taxa_carga_base"])).abs().max()
cheia = np.allclose(ca["carga_t"].values, fb.loc[ca["id_barcaca"], "capacidade"].values)
lg = log_base.copy()
lg["carga_antes"] = lg.groupby("id_barcaca")["carga_t"].shift(1)
c2 = lg[lg["estado"] == "carregando"]
e_qtd = (c2["volume_t"] - (fb.loc[c2["id_barcaca"], "capacidade"].values - c2["carga_antes"])).abs().max()
relatar("V04", "Tempos e carga a bordo",
        "Verificar as fórmulas de tempo dos dados D09 e a regra de completar o tanque.",
        f"Todos os estados do cenário-base ({len(nav)} navegações, {len(bo)} bombeios, {len(ca)} cargas).",
        "navegação = `dist_base` / `velocidade`; bombeio = volume / `vazao_bomba`; carga = volume / `taxa_carga_base` (erros ≤ 1e-8 h); após a carga, carga a bordo = capacidade; volume carregado = capacidade − carga ao chegar.",
        f"Erros máximos: navegação {e_nav:.1e} h, bombeio {e_bom:.1e} h, carga {e_car:.1e} h; tanque cheio após a carga: {cheia}; erro do volume carregado: {e_qtd:.1e} t.",
        max(e_nav, e_bom, e_car) <= TOL and cheia and e_qtd <= 1e-6,
        "Os tempos são determinísticos e calculados como especificado; a barcaça sempre completa o tanque.")

**V04 – Tempos e carga a bordo**

- **Objetivo:** Verificar as fórmulas de tempo dos dados D09 e a regra de completar o tanque.
- **Configuração:** Todos os estados do cenário-base (6955 navegações, 3477 bombeios, 3475 cargas).
- **Esperado:** navegação = `dist_base` / `velocidade`; bombeio = volume / `vazao_bomba`; carga = volume / `taxa_carga_base` (erros ≤ 1e-8 h); após a carga, carga a bordo = capacidade; volume carregado = capacidade − carga ao chegar.
- **Obtido:** Erros máximos: navegação 2.8e-10 h, bombeio 2.7e-10 h, carga 2.7e-10 h; tanque cheio após a carga: True; erro do volume carregado: 0.0e+00 t.
- **Conclusão:** Aprovado. Os tempos são determinísticos e calculados como especificado; a barcaça sempre completa o tanque.

In [11]:
# V05 - Frequência dos destinos
p_esp = pd.Series(diag_base["p_destino"], index=terminais_base["id_terminal"])
lam_p = terminais_base["taxa_chegada"] * terminais_base["prob_compra"]
p_ref = pd.Series((lam_p / lam_p.sum()).values, index=terminais_base["id_terminal"])
cont = bo["local"].value_counts().reindex(p_esp.index, fill_value=0)
n = cont.sum()
ok_d, res = np.allclose(p_esp, p_ref), []
for t in p_esp.index:
    p = p_esp[t]
    ok_d &= abs(cont[t] / n - p) <= 3 * np.sqrt(p * (1 - p) / n)
    res.append(f"{t}: {cont[t]/n:.3f} (esperado {p:.3f})")
_, d0 = simular(frota_base, so_terminal(terminais_base, "T6"), cen(horizonte=10), 1)
relatar("V05", "Sorteio do destino",
        "Verificar que o terminal de cada viagem é sorteado com probabilidade proporcional a λ_j · `prob_compra_j`.",
        f"Frequência dos terminais nos {n} bombeios do cenário-base; caso extremo com `prob_compra` > 0 só em T6.",
        "Frequências dentro de p_j ± 3√(p_j(1−p_j)/n), com p_j = λ_j·prob_j / Σ λ·prob; no caso extremo, probabilidade 1 para T6.",
        "; ".join(res) + f". Caso extremo: p(T6) = {d0['p_destino'][-1]:.0f}.",
        ok_d and d0["p_destino"][-1] == 1,
        "Os destinos reproduzem a distribuição esperada dos pedidos.")

**V05 – Sorteio do destino**

- **Objetivo:** Verificar que o terminal de cada viagem é sorteado com probabilidade proporcional a λ_j · `prob_compra_j`.
- **Configuração:** Frequência dos terminais nos 3477 bombeios do cenário-base; caso extremo com `prob_compra` > 0 só em T6.
- **Esperado:** Frequências dentro de p_j ± 3√(p_j(1−p_j)/n), com p_j = λ_j·prob_j / Σ λ·prob; no caso extremo, probabilidade 1 para T6.
- **Obtido:** T1: 0.170 (esperado 0.170); T2: 0.239 (esperado 0.233); T3: 0.210 (esperado 0.218); T4: 0.226 (esperado 0.223); T5: 0.110 (esperado 0.108); T6: 0.045 (esperado 0.049). Caso extremo: p(T6) = 1.
- **Conclusão:** Aprovado. Os destinos reproduzem a distribuição esperada dos pedidos.

In [12]:
# V06 - Volume sorteado
ok_v, res = True, []
for _, t in terminais_base.iterrows():
    x = bo.loc[bo["local"] == t["id_terminal"], "volume_t"]
    m = (t["vl_p1"] + t["vl_p2"] + t["vl_p3"]) / 3
    ok_v &= x.min() >= t["vl_p1"] and x.max() <= t["vl_p3"] and abs(x.mean() - m) <= 3 * x.std() / np.sqrt(len(x))
    res.append(f"{t['id_terminal']}: [{x.min():.0f}; {x.max():.0f}] t, média {x.mean():.1f} (esperada {m:.1f})")
relatar("V06", "Volume do atendimento",
        "Verificar que o volume entregue segue a distribuição triangular do terminal sorteado.",
        "Volumes bombeados no cenário-base, por terminal.",
        "Volumes entre mínimo e máximo; média igual a (mín + moda + máx)/3 (3 erros-padrão).",
        "; ".join(res) + ".",
        ok_v,
        "O volume é sorteado com os parâmetros do terminal de destino.")

**V06 – Volume do atendimento**

- **Objetivo:** Verificar que o volume entregue segue a distribuição triangular do terminal sorteado.
- **Configuração:** Volumes bombeados no cenário-base, por terminal.
- **Esperado:** Volumes entre mínimo e máximo; média igual a (mín + moda + máx)/3 (3 erros-padrão).
- **Obtido:** T1: [623; 1773] t, média 1131.1 (esperada 1133.3); T2: [836; 1977] t, média 1328.4 (esperada 1333.3); T3: [410; 1176] t, média 764.9 (esperada 766.7); T4: [516; 1591] t, média 993.5 (esperada 1000.0); T5: [307; 970] t, média 634.3 (esperada 633.3); T6: [274; 871] t, média 563.5 (esperada 550.0).
- **Conclusão:** Aprovado. O volume é sorteado com os parâmetros do terminal de destino.

In [13]:
# V07 - Berços da base: capacidade e FIFO
log7, d7 = simular(pd.concat([frota_base] * 2, ignore_index=True).assign(id_barcaca=lambda d: [f"B{i+1}" for i in range(len(d))]),
                   terminais_base, cen(horizonte=30, n_bercos_base=1), 1)
m_base, m7 = ocupacao_maxima(log_base), ocupacao_maxima(log7)
B = int(cenario_base["n_bercos_base"])
fifo_ok = True
for lg_ in (log_base, log7):
    f = lg_[lg_["estado"] == "fila_base"].sort_values(["t_inicio", "t_fim"])
    c = lg_[lg_["estado"] == "carregando"]
    ini_carga = c.set_index(["id_barcaca", "viagem"])["t_inicio"]
    ordem = ini_carga.reindex(list(zip(f["id_barcaca"], f["viagem"]))).dropna()
    fifo_ok &= bool((ordem.diff().dropna() >= pd.Timedelta(0)).all())
esperas7 = int((dur(log7[log7["estado"] == "fila_base"]) > 1e-9).sum())
relatar("V07", "Berços da base",
        "Verificar que nunca há mais barcaças carregando do que berços e que a fila da base é FIFO.",
        f"Cenário-base ({B} berços, {len(frota_base)} barcaças) e cenário de estresse com 1 berço e {2*len(frota_base)} barcaças (30 dias).",
        f"Ocupação máxima ≤ {B} no cenário-base; = 1 no estresse, com espera; as barcaças iniciam a carga na ordem de chegada à fila.",
        f"Ocupação máxima: base {m_base} (monitor {diag_base['ocupacao_max']}), estresse {m7} (monitor {d7['ocupacao_max']}); chegadas com espera no estresse: {esperas7}; FIFO respeitado: {fifo_ok}.",
        m_base <= B and m7 == 1 and d7["ocupacao_max"] == 1 and esperas7 > 0 and fifo_ok,
        "A capacidade dos berços é respeitada e a fila da base é atendida por ordem de chegada.")

**V07 – Berços da base**

- **Objetivo:** Verificar que nunca há mais barcaças carregando do que berços e que a fila da base é FIFO.
- **Configuração:** Cenário-base (2 berços, 4 barcaças) e cenário de estresse com 1 berço e 8 barcaças (30 dias).
- **Esperado:** Ocupação máxima ≤ 2 no cenário-base; = 1 no estresse, com espera; as barcaças iniciam a carga na ordem de chegada à fila.
- **Obtido:** Ocupação máxima: base 2 (monitor 2), estresse 1 (monitor 1); chegadas com espera no estresse: 288; FIFO respeitado: True.
- **Conclusão:** Aprovado. A capacidade dos berços é respeitada e a fila da base é atendida por ordem de chegada.

In [14]:
# V08 - Frota heterogênea
fh = frota_base.copy()
k = fh["id_barcaca"].iloc[-1]
fh.loc[fh["id_barcaca"] == k, ["velocidade", "vazao_bomba"]] = fh.loc[fh["id_barcaca"] == k, ["velocidade", "vazao_bomba"]].values * 2
log8, _ = simular(fh, terminais_base, cen(horizonte=60), 1)
fhb = fh.set_index("id_barcaca")
n8 = log8[log8["estado"] == "navegando"]
e8n = (dur(n8) - n8["local"].str.replace("Base", "").str.strip("-").map(dist).values / fhb.loc[n8["id_barcaca"], "velocidade"].values).abs().max()
b8 = log8[log8["estado"] == "bombeando"]
e8b = (dur(b8) - b8["volume_t"].values / fhb.loc[b8["id_barcaca"], "vazao_bomba"].values).abs().max()
viag = b8.groupby("id_barcaca").size()
outras = viag.drop(k)
prim_dest = (log8[log8["viagem"] == 1].query("estado == 'bombeando'").set_index("id_barcaca")["local"].drop(k)
             .equals(log_base[log_base["viagem"] == 1].query("estado == 'bombeando'").set_index("id_barcaca")["local"].drop(k)))
relatar("V08", "Frota heterogênea",
        "Verificar que cada barcaça usa os próprios atributos de `frota.csv` e que alterá-la não muda os sorteios das demais.",
        f"Barcaça {k} com velocidade e taxa de bombeio dobradas; 60 dias.",
        f"Tempos de cada barcaça calculados com os seus atributos (erro ≤ 1e-8 h); {k} faz mais viagens que qualquer outra; primeiro destino das demais igual ao do cenário-base.",
        f"Erros: navegação {e8n:.1e} h, bombeio {e8b:.1e} h; viagens: {viag.to_dict()}; primeiros destinos das demais inalterados: {prim_dest}.",
        max(e8n, e8b) <= TOL and viag[k] > outras.max() and prim_dest,
        "Os atributos são lidos por barcaça, o que permite simular frotas heterogêneas, uma das alternativas do estudo.")

**V08 – Frota heterogênea**

- **Objetivo:** Verificar que cada barcaça usa os próprios atributos de `frota.csv` e que alterá-la não muda os sorteios das demais.
- **Configuração:** Barcaça B4 com velocidade e taxa de bombeio dobradas; 60 dias.
- **Esperado:** Tempos de cada barcaça calculados com os seus atributos (erro ≤ 1e-8 h); B4 faz mais viagens que qualquer outra; primeiro destino das demais igual ao do cenário-base.
- **Obtido:** Erros: navegação 2.2e-16 h, bombeio 2.6e-10 h; viagens: {'B1': 140, 'B2': 141, 'B3': 143, 'B4': 224}; primeiros destinos das demais inalterados: True.
- **Conclusão:** Aprovado. Os atributos são lidos por barcaça, o que permite simular frotas heterogêneas, uma das alternativas do estudo.

In [15]:
# V09 - Continuidade e sequência dos estados
seq = ["navegando", "bombeando", "navegando", "fila_base", "carregando"]
cont_ok = seq_ok = carga_ok = True
for b, g in log_base.groupby("id_barcaca"):
    g = g.sort_values(["t_inicio", "t_fim"], kind="stable")
    cont_ok &= bool((g["t_inicio"].iloc[1:].values == g["t_fim"].iloc[:-1].values).all())
    seq_ok &= list(g["estado"]) == (seq * (len(g) // 5 + 1))[:len(g)]
    seq_ok &= bool((g.groupby("viagem").size().iloc[:-1] == 5).all())
    carga_ok &= bool(((g["carga_t"] >= -1e-9) & (g["carga_t"] <= fb.loc[b, "capacidade"] + 1e-9)).all())
fim = origem + pd.Timedelta(days=float(cenario_base["horizonte"]))
dentro = (log_base["t_inicio"] >= origem).all() and (log_base["t_fim"] <= fim).all() and (log_base["t_fim"] >= log_base["t_inicio"]).all()
relatar("V09", "Continuidade e sequência dos estados",
        "Verificar que o log de cada barcaça é contínuo, segue a ordem do ciclo e respeita o horizonte e a capacidade.",
        "Log do cenário-base, por barcaça.",
        "Fim de cada estado = início do seguinte; sequência navegando → bombeando → navegando → fila_base → carregando, 5 estados por viagem completa; 0 ≤ carga ≤ capacidade; tempos dentro do horizonte.",
        f"Contínuo: {cont_ok}; sequência correta: {seq_ok}; carga dentro dos limites: {carga_ok}; dentro do horizonte: {bool(dentro)}.",
        cont_ok and seq_ok and carga_ok and dentro,
        "O log descreve sem lacunas o ciclo definido no modelo conceitual.")

**V09 – Continuidade e sequência dos estados**

- **Objetivo:** Verificar que o log de cada barcaça é contínuo, segue a ordem do ciclo e respeita o horizonte e a capacidade.
- **Configuração:** Log do cenário-base, por barcaça.
- **Esperado:** Fim de cada estado = início do seguinte; sequência navegando → bombeando → navegando → fila_base → carregando, 5 estados por viagem completa; 0 ≤ carga ≤ capacidade; tempos dentro do horizonte.
- **Obtido:** Contínuo: True; sequência correta: True; carga dentro dos limites: True; dentro do horizonte: True.
- **Conclusão:** Aprovado. O log descreve sem lacunas o ciclo definido no modelo conceitual.

In [16]:
# V10 - Conservação do combustível
ok_c, res = True, []
for b, g in log_base.groupby("id_barcaca"):
    g = g.sort_values(["t_inicio", "t_fim"], kind="stable")
    carregado = g.loc[g["estado"] == "carregando", "volume_t"].sum()
    entregue = g.loc[g["estado"] == "bombeando", "volume_t"].sum()
    saldo = fb.loc[b, "carga_inicial"] + carregado - entregue
    ok_c &= abs(saldo - g["carga_t"].iloc[-1]) <= 1e-6 and abs(saldo - diag_base["carga_final"][b]) <= 1e-6
    res.append(f"{b}: {fb.loc[b, 'carga_inicial']:.0f} + {carregado:,.0f} − {entregue:,.0f} = {saldo:,.1f} t (log: {g['carga_t'].iloc[-1]:,.1f})")
relatar("V10", "Conservação do combustível",
        "Verificar que o combustível a bordo é conservado: nada é criado nem perdido entre carga e bombeio.",
        "Balanço por barcaça no cenário-base.",
        "carga inicial + Σ carregado − Σ bombeado = carga a bordo no último estado do log (e no estado final do modelo), com erro ≤ 1e-6 t.",
        "; ".join(res) + ".",
        ok_c,
        "Os volumes do log fecham o balanço de massa de cada barcaça.")

**V10 – Conservação do combustível**

- **Objetivo:** Verificar que o combustível a bordo é conservado: nada é criado nem perdido entre carga e bombeio.
- **Configuração:** Balanço por barcaça no cenário-base.
- **Esperado:** carga inicial + Σ carregado − Σ bombeado = carga a bordo no último estado do log (e no estado final do modelo), com erro ≤ 1e-6 t.
- **Obtido:** B1: 3000 + 865,471 − 865,471 = 3,000.0 t (log: 3,000.0); B2: 3000 + 852,271 − 852,981 = 2,289.7 t (log: 2,289.7); B3: 3000 + 856,384 − 857,896 = 1,488.1 t (log: 1,488.1); B4: 3000 + 866,627 − 866,627 = 3,000.0 t (log: 3,000.0).
- **Conclusão:** Aprovado. Os volumes do log fecham o balanço de massa de cada barcaça.

In [17]:
# V11 - Formato do log_barcacas.csv
lido = pd.read_csv("saida/log_barcacas.csv", parse_dates=["t_inicio", "t_fim"])
cols_ok = list(lido.columns) == COLUNAS_LOG
tipos_ok = (pd.api.types.is_integer_dtype(lido["rep"]) and pd.api.types.is_integer_dtype(lido["viagem"])
            and all(pd.api.types.is_datetime64_any_dtype(lido[c]) for c in ("t_inicio", "t_fim"))
            and pd.api.types.is_float_dtype(lido["carga_t"]) and pd.api.types.is_float_dtype(lido["volume_t"]))
estados_ok = set(lido["estado"]) == {"navegando", "bombeando", "fila_base", "carregando"}
vol_ok = (lido.loc[lido["estado"].isin(["navegando", "fila_base"]), "volume_t"] == 0).all()
reler = len(lido) == len(log_base) and all((lido[c] - log_base[c]).abs().max() <= pd.Timedelta(microseconds=1) for c in ("t_inicio", "t_fim"))
relatar("V11", "Formato do `log_barcacas.csv`",
        "Confirmar que o arquivo gravado segue a definição de saída do modelo conceitual 1.2.",
        "`saida/log_barcacas.csv` do cenário-base, relido do disco.",
        "10 colunas com os nomes e a ordem definidos; instantes em data-hora; apenas os 4 estados previstos; `volume_t` = 0 em `navegando` e `fila_base`; arquivo igual ao log em memória (1 µs).",
        f"Colunas: {cols_ok}; tipos: {tipos_ok}; estados: {sorted(set(lido['estado']))}; volume nulo onde previsto: {bool(vol_ok)}; releitura fiel: {bool(reler)}; {len(lido)} linhas.",
        cols_ok and tipos_ok and estados_ok and vol_ok and reler,
        "O log está conforme o modelo conceitual e pode ser usado no modelo integrado.")

**V11 – Formato do `log_barcacas.csv`**

- **Objetivo:** Confirmar que o arquivo gravado segue a definição de saída do modelo conceitual 1.2.
- **Configuração:** `saida/log_barcacas.csv` do cenário-base, relido do disco.
- **Esperado:** 10 colunas com os nomes e a ordem definidos; instantes em data-hora; apenas os 4 estados previstos; `volume_t` = 0 em `navegando` e `fila_base`; arquivo igual ao log em memória (1 µs).
- **Obtido:** Colunas: True; tipos: True; estados: ['bombeando', 'carregando', 'fila_base', 'navegando']; volume nulo onde previsto: True; releitura fiel: True; 17383 linhas.
- **Conclusão:** Aprovado. O log está conforme o modelo conceitual e pode ser usado no modelo integrado.

In [18]:
# V12 - Reprodutibilidade
a1, _ = simular(frota_base, terminais_base, cen(horizonte=30), 1)
a2, _ = simular(frota_base, terminais_base, cen(horizonte=30), 1)
b1, _ = simular(frota_base, terminais_base, cen(horizonte=30, semente=int(cenario_base["semente"]) + 1), 1)
c1, _ = simular(frota_base, terminais_base, cen(horizonte=30), 2)
ig, dif_s, dif_r = a1.equals(a2), not a1.drop(columns="rep").equals(b1.drop(columns="rep")), not a1.drop(columns="rep").equals(c1.drop(columns="rep"))
relatar("V12", "Reprodutibilidade e semente",
        "Verificar que a mesma configuração reproduz o mesmo log e que semente e replicação alteram os resultados.",
        "Quatro execuções de 30 dias: duas iguais, uma com semente + 1 e uma com a replicação 2.",
        "Logs idênticos para a mesma semente e replicação; diferentes nos outros casos.",
        f"Mesma configuração idêntica: {ig}; outra semente diferente: {dif_s}; outra replicação diferente: {dif_r}.",
        ig and dif_s and dif_r,
        "A semente de `cenario.csv` torna os resultados reproduzíveis.")

**V12 – Reprodutibilidade e semente**

- **Objetivo:** Verificar que a mesma configuração reproduz o mesmo log e que semente e replicação alteram os resultados.
- **Configuração:** Quatro execuções de 30 dias: duas iguais, uma com semente + 1 e uma com a replicação 2.
- **Esperado:** Logs idênticos para a mesma semente e replicação; diferentes nos outros casos.
- **Obtido:** Mesma configuração idêntica: True; outra semente diferente: True; outra replicação diferente: True.
- **Conclusão:** Aprovado. A semente de `cenario.csv` torna os resultados reproduzíveis.

In [19]:
# V13 - Parametrização
l2, _ = simular(frota_base.iloc[:2], terminais_base, cen(horizonte=120), 1)
l4, _ = simular(frota_base, terminais_base, cen(horizonte=120), 1)
razao_frota = (l4["estado"] == "bombeando").sum() / (l2["estado"] == "bombeando").sum()
h60, _ = simular(frota_base, terminais_base, cen(horizonte=60), 1)
razao_h = (l4["estado"] == "bombeando").sum() / (h60["estado"] == "bombeando").sum()
l3b, _ = simular(frota_base, terminais_base, cen(horizonte=120, n_bercos_base=1), 1)
esp1 = dur(l3b[l3b["estado"] == "fila_base"]).mean()
esp2 = dur(l4[l4["estado"] == "fila_base"]).mean()


def literais_numericos(codigo, acc):
    for k_ in codigo.co_consts:
        if hasattr(k_, "co_consts"):
            literais_numericos(k_, acc)
        elif isinstance(k_, (int, float)) and not isinstance(k_, bool):
            acc.add(float(k_))
    return acc


literais = set()
for fn in (carregar_parametros, Barcaca.__init__, Frota.__init__, Frota.registrar, Frota.sortear_atendimento, Frota.ciclo, Frota.simular):
    literais_numericos(fn.__code__, literais)
permitidos = {0.0, 1.0, 2.0}
relatar("V13", "Parametrização",
        "Verificar que o comportamento muda ao alterar as entradas e que o código não contém valores do cenário.",
        "(a) 2 e 4 barcaças, 120 dias; (b) horizonte de 60 e 120 dias; (c) base com 1 e 2 berços; (d) literais numéricos das funções do modelo.",
        "(a) e (b) número de atendimentos aproximadamente proporcional (razão entre 1,8 e 2,2); (c) espera média maior com 1 berço; (d) apenas constantes estruturais (0, 1 e 2 fluxos aleatórios por barcaça).",
        f"(a) razão {razao_frota:.2f}; (b) razão {razao_h:.2f}; (c) espera média {esp1:.2f} h com 1 berço e {esp2:.2f} h com 2; (d) literais: {sorted(literais)}.",
        1.8 <= razao_frota <= 2.2 and 1.8 <= razao_h <= 2.2 and esp1 > esp2 and literais <= permitidos,
        "Frota, base e horizonte são controlados pelos arquivos de entrada; o código só tem constantes estruturais.")

**V13 – Parametrização**

- **Objetivo:** Verificar que o comportamento muda ao alterar as entradas e que o código não contém valores do cenário.
- **Configuração:** (a) 2 e 4 barcaças, 120 dias; (b) horizonte de 60 e 120 dias; (c) base com 1 e 2 berços; (d) literais numéricos das funções do modelo.
- **Esperado:** (a) e (b) número de atendimentos aproximadamente proporcional (razão entre 1,8 e 2,2); (c) espera média maior com 1 berço; (d) apenas constantes estruturais (0, 1 e 2 fluxos aleatórios por barcaça).
- **Obtido:** (a) razão 1.98; (b) razão 2.02; (c) espera média 1.45 h com 1 berço e 0.13 h com 2; (d) literais: [0.0, 1.0, 2.0].
- **Conclusão:** Aprovado. Frota, base e horizonte são controlados pelos arquivos de entrada; o código só tem constantes estruturais.

In [20]:
# V14 - Comparação com o cálculo analítico
tb = terminais_base.assign(p=p_esp.values, ea=(terminais_base["vl_p1"] + terminais_base["vl_p2"] + terminais_base["vl_p3"]) / 3)
r0 = frota_base.iloc[0]
assert frota_base[["velocidade", "vazao_bomba"]].nunique().max() == 1
q = float(cenario_base["taxa_carga_base"])
ciclo_teo = (tb["p"] * (2 * tb["dist_base"] / r0["velocidade"] + tb["ea"] / r0["vazao_bomba"] + tb["ea"] / q)).sum()
cic = log_base[log_base["estado"] != "fila_base"].assign(d=lambda x: dur(x)).groupby(["id_barcaca", "viagem"]).filter(lambda g: len(g) == 4)
cic = cic.groupby(["id_barcaca", "viagem"])["d"].sum()
H = float(cenario_base["horizonte"]) * HORAS_POR_DIA
util_obs = dur(log_base[log_base["estado"] == "carregando"]).sum() / (B * H)
util_teo = len(frota_base) * H / (ciclo_teo + fila["duracao_h"].mean()) * (tb["p"] * tb["ea"]).sum() / q / (B * H)
relatar("V14", "Ciclo e utilização versus cálculo analítico",
        "Comparar o ciclo médio sem fila e a utilização dos berços da base com os valores calculados a partir dos parâmetros.",
        f"Cenário-base; ciclo teórico = Σ p_j (2 u_j / v + E[a_j] / pf + E[a_j] / q); utilização teórica = viagens esperadas × E[a] / q ÷ (B × H), com a espera média observada.",
        "Ciclo médio observado igual ao teórico (3 erros-padrão); diferença de utilização ≤ 0,02.",
        f"Ciclo: observado {cic.mean():.3f} h (±{3*cic.std()/np.sqrt(len(cic)):.3f}), teórico {ciclo_teo:.3f} h; utilização dos berços: observada {util_obs:.3f}, teórica {util_teo:.3f}.",
        abs(cic.mean() - ciclo_teo) <= 3 * cic.std() / np.sqrt(len(cic)) and abs(util_obs - util_teo) <= 0.02,
        "O desempenho simulado é coerente com o calculado analiticamente a partir dos parâmetros.")

**V14 – Ciclo e utilização versus cálculo analítico**

- **Objetivo:** Comparar o ciclo médio sem fila e a utilização dos berços da base com os valores calculados a partir dos parâmetros.
- **Configuração:** Cenário-base; ciclo teórico = Σ p_j (2 u_j / v + E[a_j] / pf + E[a_j] / q); utilização teórica = viagens esperadas × E[a] / q ÷ (B × H), com a espera média observada.
- **Esperado:** Ciclo médio observado igual ao teórico (3 erros-padrão); diferença de utilização ≤ 0,02.
- **Obtido:** Ciclo: observado 9.934 h (±0.104), teórico 9.926 h; utilização dos berços: observada 0.491, teórica 0.491.
- **Conclusão:** Aprovado. O desempenho simulado é coerente com o calculado analiticamente a partir dos parâmetros.

In [21]:
resumo_testes = pd.DataFrame(RESULTADOS)
display(resumo_testes)
print("Testes aprovados:", int((resumo_testes["Resultado"] == "Aprovado").sum()), "de", len(resumo_testes))

,Teste,Descrição,Resultado
0,V01,Arquivos de entrada,Aprovado
1,V02,Ciclo determinístico,Aprovado
2,V03,Estado inicial,Aprovado
3,V04,Tempos e carga a bordo,Aprovado
4,V05,Sorteio do destino,Aprovado
5,V06,Volume do atendimento,Aprovado
6,V07,Berços da base,Aprovado
7,V08,Frota heterogênea,Aprovado
8,V09,Continuidade e sequência dos estados,Aprovado
9,V10,Conservação do combustível,Aprovado


Testes aprovados: 14 de 14


## 6. Referências

- Costa, M. G.; Mesquita, M. A. (2023). *Minimizing total tardiness and makespan in ship fuelling operations in a maritime terminal*. Manuscrito.
- Grupo G112 (2026). *Projeto BVE - Modelo Conceitual*, versão 1.2; *Modelo computacional: Navios* (D08).
- PRO3342 (2026). *Aula 9 - Modelo 2: Barcaças*; *D09 - BVE: Modelo Computacional das Barcaças* (notebook-modelo e dados do cenário-base); *D08 - BVE: Modelo Computacional dos Navios* (dados do cenário-base).
- SimPy 4, NumPy e pandas: documentação oficial.